In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

def plot_scatter_with_top_hist(x, y, c=None, c_array_for_colorbar=None,
                               xlabel='Dihedral angle (°)', ylabel='–ICOHP (eV)',
                               xlim=(-180,180), ylim=None, bins=None,
                               vlines=None, cmap='viridis', scatter_marker='.', scatter_s=30,
                               top_hist_ylabel='Count', filename='plot_with_hist.png',
                               colorbar_label=None, vmin=None, vmax=None, figsize=(4,2.5)):
    if bins is None:
        bins = np.linspace(xlim[0], xlim[1], 37)
    fig = plt.figure(figsize=figsize)
    gs = fig.add_gridspec(nrows=2, ncols=2, height_ratios=[1,4], width_ratios=[1,0.04], hspace=0.05)
    ax_hist = fig.add_subplot(gs[0,0])
    ax_scatter = fig.add_subplot(gs[1,0], sharex=ax_hist)
    cax = fig.add_subplot(gs[1,1])
    valid = ~np.isnan(x)
    x_valid = np.asarray(x)[valid]
    counts, edges = np.histogram(x_valid, bins=bins)
    centers = (edges[:-1] + edges[1:]) / 2.0
    widths = np.diff(edges)
    ax_hist.bar(centers, counts, width=widths, align='center', color='#a7a9ac', edgecolor=None, alpha=0.7)
    ax_hist.set_ylabel(top_hist_ylabel)
    ax_hist.tick_params(axis='x', labelbottom=False)
    ax_hist.set_xlim(xlim)
    if c is not None:
        sc = ax_scatter.scatter(x, y, c=c, cmap=cmap, marker=scatter_marker, s=scatter_s,
                                vmin=vmin, vmax=vmax)
    else:
        sc = ax_scatter.scatter(x, y, marker=scatter_marker, s=scatter_s)
    draw_colorbar = (c is not None) or (c_array_for_colorbar is not None and len(np.atleast_1d(c_array_for_colorbar)) > 0)
    if draw_colorbar:
        from matplotlib import cm
        from matplotlib.colors import Normalize
        if c_array_for_colorbar is not None:
            arr = np.asarray(c_array_for_colorbar)
            arr = arr[~np.isnan(arr)] if arr.size else arr
            arr_min = np.nanmin(arr) if arr.size else None
            arr_max = np.nanmax(arr) if arr.size else None
        else:
            arr_min, arr_max = None, None
        norm = Normalize(vmin=vmin if vmin is not None else (arr_min if arr_min is not None else 0.0),
                         vmax=vmax if vmax is not None else (arr_max if arr_max is not None else 1.0))
        mappable = cm.ScalarMappable(norm=norm, cmap=cm.get_cmap(cmap))
        mappable.set_array([])
        cbar = fig.colorbar(mappable, cax=cax, orientation='vertical')
        if colorbar_label:
            cbar.set_label(colorbar_label)
    ax_scatter.set_xlabel(xlabel)
    ax_scatter.set_ylabel(ylabel)
    if ylim is not None:
        ax_scatter.set_ylim(ylim)
    ax_scatter.set_xlim(xlim)
    ax_scatter.tick_params(axis='both', which='major', direction='in')
    if vlines:
        for vx in vlines:
            ax_scatter.axvline(x=vx, color='k', linestyle='--', linewidth=0.7)
            ax_hist.axvline(x=vx, color='k', linestyle='--', linewidth=0.7)
    fig.savefig(filename, bbox_inches='tight', dpi=300)
    plt.close(fig)


def nanmean(series):
    arr = pd.to_numeric(series, errors='coerce').to_numpy(dtype=float)
    return float(np.nanmean(arr)) if arr.size else np.nan


if __name__ == "__main__":
    bonding_df = pd.read_csv('bonding_df.csv', index_col=0)
    crystal_files = {
        'blackAs': 'blackAs_icohps.csv',
        'greyAs' : 'greyAs_icohps.csv',
        'yellowAs': 'yellowAs_icohps.csv',
        'violetAs': 'violetAs_icohps.csv',
        'fibrousAs': 'fibrousAs_icohps.csv'
    }
    crystals = {}
    for k, v in crystal_files.items():
        p = Path(v)
        if p.exists():
            df = pd.read_csv(v, index_col=0)
            crystals[k] = df

    bonding_df.to_csv('bonding_df_saved_for_plots.csv')

    fig, ax = plt.subplots(figsize=(4,2.5))
    ax.tick_params(axis='both', which='major', direction='in')
    ax.set_xlabel('As-As distance (A)')
    ax.set_ylabel('Normalized -ICOHP')
    ax.set_ylim(0.1,1.1)
    ax.set_xlim(2.3,2.95)

    ax.plot(bonding_df['bond_dist'], bonding_df['icohp_scaled'],
            color="#C17F91", marker='X', markersize=5, linewidth=0, label='a-As')

    cmap_markers = {
        'blackAs': ('#000000','o'),
        'greyAs' : ('#929292','s'),
        'yellowAs':('#f2d600','^'),
        'violetAs':('#a000a0','D'),
        'fibrousAs':('#53006D','v')
    }
    for name, df in crystals.items():
        col, marker = cmap_markers.get(name, ('#000000','o'))
        ax.plot(df['bond_dist'], df['icohp_scaled'],
                color=col, marker=marker, markersize=5, fillstyle='none', linewidth=0, label=name)

    mean_aAs = nanmean(bonding_df['icohp_scaled'])
    print(f"Average normalized -ICOHP (a-As):   {mean_aAs:.4f}")
    if 'blackAs' in crystals:
        mean_black = nanmean(crystals['blackAs']['icohp_scaled'])
        print(f"Average normalized -ICOHP (black As): {mean_black:.4f}")

    ax.legend()
    plt.savefig('rvv10relabeled_icohpdist.png', bbox_inches='tight', dpi=1000)
    plt.close()

    short_mask = bonding_df['bond_dist'] < 2.45
    medium_mask = (bonding_df['bond_dist'] >= 2.45) & (bonding_df['bond_dist'] <= 2.55)
    long_mask  = bonding_df['bond_dist'] > 2.55

    print(f"Short-bond points (<2.45 A): {short_mask.sum()}")
    print(f"Medium-bond points (2.45-2.55 A): {medium_mask.sum()}")
    print(f"Long-bond points (>2.55 A): {long_mask.sum()}")

    all_angles = bonding_df['dih_angle'].dropna().to_numpy()
    if all_angles.size:
        angle_min, angle_max = np.nanmin(all_angles), np.nanmax(all_angles)
    else:
        angle_min, angle_max = -180, 180
    pad = 5.0
    xlim = (angle_min - pad, angle_max + pad)
    bins = np.linspace(xlim[0], xlim[1], 37)

    color_col = 'bond_dist'
    if color_col in bonding_df.columns:
        color_vmin = float(np.nanmin(bonding_df[color_col].to_numpy()))
        color_vmax = float(np.nanmax(bonding_df[color_col].to_numpy()))
    else:
        color_vmin, color_vmax = None, None

    if short_mask.sum() > 0:
        x_short = bonding_df.loc[short_mask, 'dih_angle']
        y_short = -bonding_df.loc[short_mask, 'icohp_value']
        c_short = bonding_df.loc[short_mask, 'bond_dist']
        plot_scatter_with_top_hist(
            x=x_short,
            y=y_short,
            c=c_short,
            c_array_for_colorbar=bonding_df['bond_dist'],
            xlabel='Dihedral angle (°)',
            ylabel='-ICOHP (eV)',
            xlim=xlim,
            ylim=(2.4, 5.1),
            bins=bins,
            vlines=[55, -55],
            cmap='viridis',
            scatter_marker='.',
            scatter_s=30,
            top_hist_ylabel='Count',
            filename='angleicohp_short_distance<2.45_withhist.png',
            colorbar_label='As-As distance (A)',
            vmin=color_vmin,
            vmax=color_vmax,
            figsize=(4,2.5)
        )

    if medium_mask.sum() > 0:
        x_med = bonding_df.loc[medium_mask, 'dih_angle']
        y_med = -bonding_df.loc[medium_mask, 'icohp_value']
        c_med = bonding_df.loc[medium_mask, 'bond_dist']
        plot_scatter_with_top_hist(
            x=x_med,
            y=y_med,
            c=c_med,
            c_array_for_colorbar=bonding_df['bond_dist'],
            xlabel='Dihedral angle (°)',
            ylabel='-ICOHP (eV)',
            xlim=xlim,
            ylim=(2.4, 5.1),
            bins=bins,
            vlines=[55, -55],
            cmap='viridis',
            scatter_marker='.',
            scatter_s=30,
            top_hist_ylabel='Count',
            filename='angleicohp_medium_2.45<=distance<=2.55_withhist.png',
            colorbar_label='As-As distance (A)',
            vmin=color_vmin,
            vmax=color_vmax,
            figsize=(4,2.5)
        )

    if long_mask.sum() > 0:
        x_long = bonding_df.loc[long_mask, 'dih_angle']
        y_long = -bonding_df.loc[long_mask, 'icohp_value']
        c_long = bonding_df.loc[long_mask, 'bond_dist']
        plot_scatter_with_top_hist(
            x=x_long,
            y=y_long,
            c=c_long,
            c_array_for_colorbar=bonding_df['bond_dist'],
            xlabel='Dihedral angle (°)',
            ylabel='-ICOHP (eV)',
            xlim=xlim,
            ylim=(2.4, 5.1),
            bins=bins,
            vlines=[55, -55],
            cmap='viridis',
            scatter_marker='.',
            scatter_s=30,
            top_hist_ylabel='Count',
            filename='angleicohp_long_distance>2.55_withhist.png',
            colorbar_label='As-As distance (A)',
            vmin=color_vmin,
            vmax=color_vmax,
            figsize=(4,2.5)
        )

    x_all = bonding_df['dih_angle']
    y_all = -bonding_df['icohp_value']
    c_all = bonding_df['bond_dist']
    plot_scatter_with_top_hist(
        x=x_all,
        y=y_all,
        c=c_all,
        c_array_for_colorbar=bonding_df['bond_dist'],
        xlabel='Dihedral angle (°)',
        ylabel='-ICOHP (eV)',
        xlim=xlim,
        ylim=None,
        bins=bins,
        vlines=[55, -55],
        cmap='viridis',
        scatter_marker='.',
        scatter_s=30,
        top_hist_ylabel='Count',
        filename='rvv10relabeled_angleicohp_withhist.png',
        colorbar_label='As-As distance (A)',
        vmin=color_vmin,
        vmax=color_vmax,
        figsize=(4,2.5)
    )